# 序列到序列学习

## 编码器

编码器的核心任务是接收一个长度可变的输入序列，并将其转换成一个形状固定的上下文变量 $\mathbf{c}$

将词元$x_t$的输入特征向量$ x_t, h_{t-1}$转换为$h_t$:
$$h_t = f(x_t, h_{t-1})$$

生成上下文变量$c$,编码器会通过一个选定的函数 $q$，将整个序列产生的所有时间步的隐状态（从 $h_1$ 到 $h_T$）整合，最终转换为唯一的上下文变量 $\mathbf{c}$。

$$\mathbf{c} = q(h_1, \dots, h_T)$$

### 嵌入层

$vocab\_size \times embed\_size$的权重矩阵

### Bi-RNN
* 前向隐状态 ($\overrightarrow{H}_t$): $\overrightarrow{H}_t = \phi(X_t W_{xh}^{(f)} + \overrightarrow{H}_{t-1} W_{hh}^{(f)} + b_h^{(f)})$。
* 反向隐状态($\overleftarrow{H}_t$)：$\overleftarrow{H}_t = \phi(X_t W_{xh}^{(b)} + \overleftarrow{H}_{t+1} W_{hh}^{(b)} + b_h^{(b)})$。

这里因为能看到全部的文本，所以可以使用双向的RNN网络

### 代码实现

两层门控单元实现循环编码器

GRU单元：
* 重置门：决定模型保留多少过去的信息
$$R_t = \sigma(X_t W_{xr} + H_{t-1} W_{hr} + b_r)$$
* 更新门：决定模型的隐状态中会保留多少旧状态
$$Z_t = \sigma(X_t W_{xz} + H_{t-1} W_{hz} + b_z)$$

In [3]:
import collections
import math
import torch
from torch import nn
from d2l import torch as d2l

class Seq2SeqEncoder(d2l.Encoder):
    """用于序列到序列学习的循环神经网络编码器"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        super(Seq2SeqEncoder, self).__init__(**kwargs)  # 调用父类的初始化方法
        # 嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size, num_hiddens, num_layers,
                          dropout=dropout)

    def forward(self, X, *args):
        # 输出'X'的形状：(batch_size,num_steps,embed_size)
        X = self.embedding(X)
        # 在循环神经网络模型中，第一个轴对应于时间步
        X = X.permute(1, 0, 2)  # (batch_size,num_steps,embed_size)-->(num_steps, batch_size, embed_size)
        # 如果未提及状态，则默认为0
        output, state = self.rnn(X)
        # output的形状:(num_steps,batch_size,num_hiddens)
        # state的形状:(num_layers,batch_size,num_hiddens)
        return output, state

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


实例化上述编码器的实现： 使用一个两层GRU，隐藏单元数是16，给定一个小批量的输入序列X（批量大小为4，时间步长为7）

In [4]:
encoder = Seq2SeqEncoder(vocab_size=10, embed_size=8, num_hiddens=16,
                         num_layers=2)
encoder.eval()  # 设置为评估模式，不适用dropout
X = torch.zeros((4, 7), dtype=torch.long)
output, state = encoder(X)
output.shape
state.shape

torch.Size([2, 4, 16])

* output的形状：网络最顶层所在的所有时间步生成的隐状态$H_t$:(num_steps,batch_size,num_hiddens)
* state的形状：所有的隐藏层在最后一个时间步生成的隐状态并最后进行堆叠： (num_layers, batch_size, num_hiddens)

output: 取最后一层，保留所有的时间步；

state：取最后一个时间步，保留所有层。

## 解码器

对于每一个时间步，解码器输出的$y_{\hat t}$取决于前面输出的子序列和编码器得到的上下文变量$c$,条件概率可以写成：$P(y_t \mid y_1,\dots,y_{t-1},c)$

对于解码器的隐藏状态可以写成：
$$
s_t = g(y_{t-1}, c, s_{t-1})
$$

* $(s_t)$：解码器当前时刻的隐藏状态；

* $(s_{t-1})$：上一时刻的隐藏状态；

* $(y_{t-1})$：上一个输入词；

* $(c)$：编码器提供的上下文。

当实现解码器时， 我们直接使用编码器最后一个时间步的隐状态来初始化解码器的隐状态;循环神经网络实现的编码器和解码器具有相同数量的层和隐藏单元

上下文变量在所有的时间步与与解码器的输入进行拼接，在循环神经网络的最后一层使用全连接层来变换隐状态（预测输出词元的概率分布）

代码：

In [5]:
class Seq2SeqDecoder(d2l.Decoder):
    """用于序列到序列学习的循环神经网络解码器"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        super(Seq2SeqDecoder, self).__init__(**kwargs)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size + num_hiddens, num_hiddens, num_layers,
                          dropout=dropout)  # 送给GRU的包括除了嵌入层的embed_size,还有编码器提供的context
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, *args):
        return enc_outputs[1]

    def forward(self, X, state):
        # 输出'X'的形状：(batch_size,num_steps,embed_size)
        X = self.embedding(X).permute(1, 0, 2)
        # 广播context，使其具有与X相同的num_steps
        context = state[-1].repeat(X.shape[0], 1, 1)   # 重复X.shape[0]次，即每个num_steps都要能看到这个context，维度由(batch_size, num_hiddens)-> (num_steps, batch_size, num_hiddens)
        X_and_context = torch.cat((X, context), 2)  # 在最后一个维度进行拼接
        output, state = self.rnn(X_and_context, state)
        output = self.dense(output).permute(1, 0, 2)
        # output的形状:(batch_size,num_steps,vocab_size)
        # state的形状:(num_layers,batch_size,num_hiddens)
        return output, state

实例化解码器

In [ ]:
# X.shape:(4, 7)
decoder = Seq2SeqDecoder(vocab_size=10, embed_size=8, num_hiddens=16,
                         num_layers=2)
decoder.eval()
state = decoder.init_state(encoder(X))
output, state = decoder(X, state)
output.shape, state.shape

(torch.Size([4, 7, 10]), torch.Size([2, 4, 16]))

## 损失函数

定义一个带mask的softmax损失函数，填充词元不参与loss的计算

在多分类任务中，单个时间步的模型预测概率分布与真实独热标签之间的交叉熵损失基础公式为：

$$l(\mathbf{y}, \mathbf{\hat{y}}) = - \sum_{j=1}^{q} y_j \log(\hat{y}_j)$$

引入掩码权重变量：$w_t$

$$L = \sum_{t=1}^T w_t l(\mathbf{y}_t, \mathbf{\hat{y}}_t)$$

代码实现

In [7]:

def sequence_mask(X, valid_len, value=0):
    """在序列中屏蔽不相关的项"""
    maxlen = X.size(1)
    mask = torch.arange((maxlen), dtype=torch.float32,
                        device=X.device)[None, :] < valid_len[:, None]  # 对维度进行扩展，广播机制
    X[~mask] = value  # 把无效的位置，改成value
    return X

X = torch.tensor([[1, 2, 3], [4, 5, 6]])
sequence_mask(X, torch.tensor([1, 2]))

tensor([[1, 0, 0],
        [4, 5, 0]])

指定时间步上的值

In [9]:
X = torch.ones(2, 3, 4)
sequence_mask(X, torch.tensor([1, 2]), value=-1)  # 样本一只保留第一个时间步，样本二只保留第二个

tensor([[[ 1.,  1.,  1.,  1.],
         [-1., -1., -1., -1.],
         [-1., -1., -1., -1.]],

        [[ 1.,  1.,  1.,  1.],
         [ 1.,  1.,  1.,  1.],
         [-1., -1., -1., -1.]]])

使用softmax来遮蔽不相关的预测

先算每个时间步的交叉熵，再把 padding 位置的损失清零，最后对每个样本取平均。

In [10]:
#@save
class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    """带遮蔽的softmax交叉熵损失函数"""
    # pred的形状：(batch_size,num_steps,vocab_size)
    # label的形状：(batch_size,num_steps)
    # valid_len的形状：(batch_size,)
    def forward(self, pred, label, valid_len):
        weights = torch.ones_like(label)
        weights = sequence_mask(weights, valid_len)
        self.reduction='none'
        unweighted_loss = super(MaskedSoftmaxCELoss, self).forward(
            pred.permute(0, 2, 1), label)  # 这里的输出是(batch_size, num_steps)
        weighted_loss = (unweighted_loss * weights).mean(dim=1)  # 沿着时间步去求取平均
        return weighted_loss

unweighted_loss = super(MaskedSoftmaxCELoss, self).forward(pred.permute(0, 2, 1), label) 

这个permute不会影响unweight_loss的形状，vocab_size在计算交叉熵的时候被消去了，得到的形状是(batch_size, num_steps)

测试

In [11]:
loss = MaskedSoftmaxCELoss()
loss(torch.ones(3, 4, 10), torch.ones((3, 4), dtype=torch.long),
     torch.tensor([4, 2, 0]))

tensor([2.3026, 1.1513, 0.0000])

## 训练

主要多的是teacher_forcing部分

In [ ]:
def train_seq2seq(net, data_iter, lr, num_epochs, tgt_vocab, device):  # tgt_vocab: target language的词汇表
    """训练序列到序列模型"""
    def xavier_init_weights(m):
        if type(m) == nn.Linear:
            nn.init.xavier_uniform_(m.weight)
        if type(m) == nn.GRU:
            for param in m._flat_weights_names:
                if "weight" in param:
                    nn.init.xavier_uniform_(m._parameters[param])  

    net.apply(xavier_init_weights)
    net.to(device)

    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss = MaskedSoftmaxCELoss()

    net.train()  # 训练模式，启动dropout层
    animator = d2l.Animator(
        xlabel='epoch',
        ylabel='loss',
        xlim=[10, num_epochs]
    )

    for epoch in range(num_epochs):
        timer = d2l.Timer()
        metric = d2l.Accumulator(2)

        for batch in data_iter:
            optimizer.zero_grad()

            X, X_valid_len, Y, Y_valid_len = [
                x.to(device) for x in batch
            ]  # 分别对应源语言输入序列，源语言有效长度，target language输入序列，target language有效长度

# teacher forcing阶段
            bos = torch.tensor(
                [tgt_vocab['<bos>']] * Y.shape[0],  # target language的batch_size个<bos> token
            # 每一个句子的开头都要插入一个<bos>开始符
                device=device
            ).reshape(-1, 1)  # 这里reshape是为了和Y[:, :-1]拼接

            dec_input = torch.cat(
                [bos, Y[:, :-1]],  # 这个地方是保留Y的所有行，去掉Y的最后一列
                1  # 按列拼接
            )  

            Y_hat, _ = net(
                X,
                dec_input,  # 训练的时候，不让decoder吃自己的输出，而是把真实的标签传给他
                X_valid_len
            )  # 这个net是encoder-decoder模型

            l = loss(
                Y_hat,
                Y,
                Y_valid_len
            )

            l.sum().backward()

            d2l.grad_clipping(net, 1)

            num_tokens = Y_valid_len.sum()

            optimizer.step()

            with torch.no_grad():
                metric.add(
                    l.sum(),
                    num_tokens
                )

### teacher forcing

特定的序列开始词元（“<bos>”）和 原始的输出序列（不包括序列结束词元“<eos>”） 拼接在一起作为解码器的输入

```
bos = torch.tensor(
                [tgt_vocab['<bos>']] * Y.shape[0],
                device=device
            ).reshape(-1, 1)

dec_input = torch.cat(
    [bos, Y[:, :-1]],
    1
)

Y_hat, _ = net(
                X,
                dec_input,  
                X_valid_len
)
```

### 创建并训练一个encoder-decoder模型用于序列到序列的学习

In [1]:
embed_size, num_hiddens, num_layers, dropout = 32, 32, 2, 0.1
batch_size, num_steps = 64, 10
lr, num_epochs, device = 0.005, 300, d2l.try_gpu()

train_iter, src_vocab, tgt_vocab = d2l.load_data_nmt(batch_size, num_steps)
encoder = Seq2SeqEncoder(len(src_vocab), embed_size, num_hiddens, num_layers,
                        dropout)
decoder = Seq2SeqDecoder(len(tgt_vocab), embed_size, num_hiddens, num_layers,
                        dropout)
net = d2l.EncoderDecoder(encoder, decoder)   # 网络是一个encoder-decoder模型，用于序列到序列的学习
train_seq2seq(net, train_iter, lr, num_epochs, tgt_vocab, device)

NameError: name 'd2l' is not defined

## 预测

解码器的输入全部来自于前一时间步的预测token

In [ ]:
#@save
def predict_seq2seq(net, src_sentence, src_vocab, tgt_vocab, num_steps,
                    device, save_attention_weights=False):
    """序列到序列模型的预测"""
    # 在预测时将net设置为评估模式
    net.eval()
    src_tokens = src_vocab[src_sentence.lower().split(' ')] + [
        src_vocab['<eos>']]  # 处理源句子，添加序列结束词元
    enc_valid_len = torch.tensor([len(src_tokens)], device=device)
    src_tokens = d2l.truncate_pad(src_tokens, num_steps, src_vocab['<pad>'])  # 输入统一为num_steps
    # 添加批量轴
    enc_X = torch.unsqueeze(
        torch.tensor(src_tokens, dtype=torch.long, device=device), dim=0)  # 添加一个batch维 变为(1, num_steps )
    enc_outputs = net.encoder(enc_X, enc_valid_len)
    dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
    # 添加批量轴
    dec_X = torch.unsqueeze(torch.tensor(
        [tgt_vocab['<bos>']], dtype=torch.long, device=device), dim=0)  # 初始化输出序列为开始token
    output_seq, attention_weight_seq = [], []  # attention_weight_seq是为后面留的接口
    for _ in range(num_steps):
        Y, dec_state = net.decoder(dec_X, dec_state)  # 得到的Y：（batch_size, num_steps, vocab_size）
        # 我们使用具有预测最高可能性的词元，作为解码器在下一时间步的输入
        dec_X = Y.argmax(dim=2)  # 这里是使用解码器的输出作为下一个时间步的输入
        pred = dec_X.squeeze(dim=0).type(torch.int32).item()  # squeeze（0）将batch_size维压缩为1，得到一个标量
        # 保存注意力权重（稍后讨论）
        if save_attention_weights:
            attention_weight_seq.append(net.decoder.attention_weights)
        # 一旦序列结束词元被预测，输出序列的生成就完成了
        if pred == tgt_vocab['<eos>']:
            break
        output_seq.append(pred)
    return ' '.join(tgt_vocab.to_tokens(output_seq)), attention_weight_seq

## 评估指标

$$
\mathrm{BLEU}
=
\exp\left(
\min\left(
0,\,
1-\frac{\mathrm{len}_{\mathrm{label}}}{\mathrm{len}_{\mathrm{pred}}}
\right)
\right)
\prod_{n=1}^{k}
p_n^{1/2^n}
$$

* 长度惩罚项：
$\exp\left(
\min\left(
0,\,
1-\frac{\mathrm{len}_{\mathrm{label}}}{\mathrm{len}_{\mathrm{pred}}}
\right)
\right)$

* n-gram匹配项：$\prod_{n=1}^{k}
p_n^{1/2^n}$  n越大，匹配的程度越高，越接近1（给的权重比较大）

越接近1，匹配程度越高